# MiniMax H3 na Colab L4 — H3-Image (do 16 MP) + H3-Video (I2V) + tunel

Reprodukcja flow z tweeta [@hiwise_aigc](https://x.com/hiwise_aigc/status/2122547991894737058) przeniesiona z RTX 5060 8 GB na Colab L4.

**Flow z tweeta:** skrypt pisze LLM → H3-Image generuje klatki w natywnej rozdzielczości → H3-Video animuje każdą klatkę (I2V) → FFmpeg skleja 10 ujęć w 20 s film.
Autor: RTX 5060 8 GB + 32 GB RAM, 9 min 12 s na całość, peak VRAM 7556 / 8151 MiB, modele kwantyzowane.

**Ten notebook (Colab L4, zmierzone na tej instancji):** 23 034 MiB VRAM (22,49 GiB użyteczne), 52 GB RAM, 236 GB dysku, Python 3.13.16, driver 580.82.07 — czyli ~2,8× VRAM i 1,6× RAM względem autora, więc ten sam pipeline wchodzi bez kombinowania z offloadem.

Zestaw domyślny jest **odcenzurowany**: tekst encoder to Sakura-MiniMax-H3-Uncensored (Q4_K_M, 15,08 GB), a denoisery H3 w wersji GGUF/int8. Wariant `l4-standard` podmienia encoder na stockowy, `l4-nsfw-video` dokłada denoiser 10Eros-Max.

**Koszt:** L4 liczy 1,94 CU/h → 446 CU ≈ 230 h GPU. Pamiętaj o ostatniej komórce (zwolnienie sesji), bo bezczynna sesja zjada jednostki.

In [ ]:
# ── 1. Co dostaliśmy od Colaba ────────────────────────────────────────────────
import subprocess, os, shutil

def sh(cmd):
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip())

sh("nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv")
sh("df -h /content | tail -1")
sh("free -g | head -2")
print("python:", subprocess.run(["python","-V"],capture_output=True,text=True).stdout.strip())

In [ ]:
%%bash
set -euo pipefail
cd /content

# ComfyUI (0.45.0+ wymagane przez szablony MiniMax-H3-Image)
if [ ! -d ComfyUI ]; then
  git clone --depth 1 https://github.com/comfyanonymous/ComfyUI
fi
cd ComfyUI
python -c "import comfyui_version as v; print('ComfyUI', v.__version__)"
pip install -q -r requirements.txt
pip install -q hf_xet

# ComfyUI-GGUF — loadery dla kwantyzowanych DiT i text encoderów
mkdir -p custom_nodes
if [ ! -d custom_nodes/ComfyUI-GGUF ]; then
  git clone --depth 1 https://github.com/city96/ComfyUI-GGUF custom_nodes/ComfyUI-GGUF
fi
pip install -q -r custom_nodes/ComfyUI-GGUF/requirements.txt

# cloudflared — szybki tunel bez konta i bez tokenu
if [ ! -x /usr/local/bin/cloudflared ]; then
  curl -sL -o /usr/local/bin/cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
  chmod +x /usr/local/bin/cloudflared
fi
cloudflared --version

In [ ]:
import os, time
from concurrent.futures import ThreadPoolExecutor, as_completed

# SET: "l4-nsfw" (domyślny) | "l4-standard" | "l4-nsfw-video" | "l4-nsfw-gguf"
SET = "l4-nsfw"

COMFY = "/content/ComfyUI"

# Xet zamiast hf_transfer (HF_HUB_ENABLE_HF_TRANSFER jest deprecated i już nieużywane).
# HF_XET_HIGH_PERFORMANCE = więcej równoległych strumieni + agresywniejszy cache chunków.
os.system("pip install -q hf_xet 2>/dev/null")   # no-op jeśli już jest
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)

SETS = {
    "l4-nsfw": [
        ("Abiray/MiniMax-H3-Image-GGUF", "MiniMax-H3-Image-Pruned-Q5_K_M.gguf", "diffusion_models"),
        ("webmp3/Sakura-MiniMax-H3-Uncensored-GGUF",
         "Sakura-MiniMax-H3-TextEncoder-Uncensored-Q4_K_M-14.05GiB.gguf", "text_encoders"),
        ("Comfy-Org/MiniMax-H3-Image", "vae/minimax_h3_image_vae_fp16.safetensors", "vae"),
        ("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "diffusion_models"),
        ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_int8_convrot.safetensors", "vae"),
        ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
        ("lightx2v/Minimax-h3-Turbo", "minimax_h3_fl2v_turbo_8step_v1.2_768p_comfyui_bf16.safetensors", "loras"),
    ],
    "l4-standard": [
        ("unsloth/MiniMax-H3-Image-GGUF", "minimax_h3_image_pruned-Q6_K.gguf", "diffusion_models"),
        ("joeygambino/MiniMax-H3-encoder-GGUF", "MiniMax-H3-encoder-Q4_K_M.gguf", "text_encoders"),
        ("Comfy-Org/MiniMax-H3-Image", "vae/minimax_h3_image_vae_fp16.safetensors", "vae"),
        ("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "diffusion_models"),
        ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_int8_convrot.safetensors", "vae"),
        ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
        ("lightx2v/Minimax-h3-Turbo", "minimax_h3_fl2v_turbo_8step_v1.2_768p_comfyui_bf16.safetensors", "loras"),
    ],
    "l4-nsfw-video": [
        ("Abiray/MiniMax-H3-Image-GGUF", "MiniMax-H3-Image-Pruned-Q5_K_M.gguf", "diffusion_models"),
        ("webmp3/Sakura-MiniMax-H3-Uncensored-GGUF",
         "Sakura-MiniMax-H3-TextEncoder-Uncensored-Q4_K_M-14.05GiB.gguf", "text_encoders"),
        ("Comfy-Org/MiniMax-H3-Image", "vae/minimax_h3_image_vae_fp16.safetensors", "vae"),
        ("OliviaRossi/Minimax-h3_Singularity-x-10Eros-Max_PRISM",
         "Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors", "diffusion_models"),
        ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_int8_convrot.safetensors", "vae"),
        ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
        ("lightx2v/Minimax-h3-Turbo", "minimax_h3_fl2v_turbo_8step_v1.2_768p_comfyui_bf16.safetensors", "loras"),
    ],
    # Fallback przy OOM na wideo: DiT Q4_K_M (19,86 GB) zamiast int8 (20,97 GB).
    "l4-nsfw-gguf": [
        ("Abiray/MiniMax-H3-Image-GGUF", "MiniMax-H3-Image-Pruned-Q5_K_M.gguf", "diffusion_models"),
        ("webmp3/Sakura-MiniMax-H3-Uncensored-GGUF",
         "Sakura-MiniMax-H3-TextEncoder-Uncensored-Q4_K_M-14.05GiB.gguf", "text_encoders"),
        ("Comfy-Org/MiniMax-H3-Image", "vae/minimax_h3_image_vae_fp16.safetensors", "vae"),
        ("Abiray/MiniMax-H3-GGUF", "unet/MiniMax-H3-FL2VA-Q4_K_M.gguf", "diffusion_models"),
        ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_int8_convrot.safetensors", "vae"),
        ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
        ("lightx2v/Minimax-h3-Turbo", "minimax_h3_fl2v_turbo_8step_v1.2_768p_comfyui_bf16.safetensors", "loras"),
    ],
}

from huggingface_hub import hf_hub_download

def fetch(repo, filename, subdir):
    models = os.path.join(COMFY, "models")
    dst = os.path.join(models, subdir, os.path.basename(filename))
    if os.path.exists(dst) and os.path.getsize(dst) > 300 * 1024**2:
        return dst, 0.0, True
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    t0 = time.time()
    got = hf_hub_download(repo_id=repo, filename=filename, local_dir=models)
    if os.path.abspath(got) != os.path.abspath(dst):
        os.replace(got, dst)
    return dst, time.time() - t0, False

todo = SETS[SET]
print(f"SET={SET}: {len(todo)} plików, równolegle (max 6 strumieni)\n")
t_all = time.time()
total = 0

with ThreadPoolExecutor(max_workers=min(6, len(todo))) as ex:
    futs = {ex.submit(fetch, *t): t for t in todo}
    for f in as_completed(futs):
        dst, dt, cached = f.result()
        size = os.path.getsize(dst)
        total += size
        tag = "cache" if cached else f"{dt:5.1f}s  {size/1e6/max(dt, 0.001):5.0f} MB/s"
        print(f"[ok] {os.path.relpath(dst, COMFY):64s} {size/1e9:6.2f} GB  {tag}")

wall = time.time() - t_all
print(f"\nrazem {total/1e9:.2f} GB w {wall:.0f}s → {total/1e6/max(wall, 0.001):.0f} MB/s średnio")
os.system("df -h /content | tail -1")

In [ ]:
# ── 3. Szablony workflow (do ręcznego użycia w UI) ───────────────────────────
import urllib.request
WF = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF, exist_ok=True)
TEMPLATES = {
    "image_minimax_h3_image_t2i.json":  "templates/image_minimax_h3_image_t2i.json",
    "image_minimax_h3_image_edit.json": "templates/image_minimax_h3_image_edit.json",
    "video_minimax_h3_i2v.json":        "templates/video_minimax_h3_i2v.json",
    "video_minimax_h3_r2v.json":        "templates/video_minimax_h3_r2v.json",
}
for name, path in TEMPLATES.items():
    url = "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/" + path
    dst = os.path.join(WF, name)
    urllib.request.urlretrieve(url, dst)
    print(f"[ok] {name:42s} {os.path.getsize(dst)/1024:.0f} kB")

In [ ]:
# ── 4. Start ComfyUI ─────────────────────────────────────────────────────────
import subprocess, time, urllib.request, os, signal

LOG = "/content/comfy.log"
if os.path.exists("/content/comfy.pid"):
    try:
        os.kill(int(open("/content/comfy.pid").read()), signal.SIGTERM)
        time.sleep(3)
    except Exception:
        pass

cmd = ("cd /content/ComfyUI && nohup python main.py "
       "--listen 127.0.0.1 --port 8188 --reserve-vram 0.3 "
       f"> {LOG} 2>&1 & echo $! > /content/comfy.pid")
subprocess.run(cmd, shell=True, check=True)

ok = False
for i in range(90):
    time.sleep(2)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3) as r:
            print("ComfyUI gotowe po", (i+1)*2, "s")
            print(r.read().decode()[:600])
            ok = True
            break
    except Exception:
        pass

if not ok:
    print("=== ostatnie 40 linii logu ===")
    print(subprocess.run(f"tail -40 {LOG}", shell=True, capture_output=True, text=True).stdout)
    raise RuntimeError("ComfyUI nie wstało na :8188")

In [ ]:
# ── 5. Smoke test: czy wszystkie potrzebne nody są zarejestrowane ────────────
import json, urllib.request

with urllib.request.urlopen("http://127.0.0.1:8188/object_info", timeout=60) as r:
    info = json.load(r)

need = ["UnetLoaderGGUF", "CLIPLoaderGGUF", "UNETLoader", "CLIPLoader", "VAELoader",
        "KSampler", "MiniMaxH3ImageConditioning", "MiniMaxH3ReferenceToVideo",
        "SaveImage", "SaveVideo", "LoraLoaderModelOnly"]
missing = [n for n in need if n not in info]
print("nody OK:", len(need) - len(missing), "/", len(need))
print("brakuje:", missing if missing else "—")

def files(node, field):
    try:
        return info[node]["input"]["required"][field][0]
    except Exception:
        return []

print("\nGGUF DiT widziane przez ComfyUI:", files("UnetLoaderGGUF", "unet_name"))
print("GGUF CLIP widziane przez ComfyUI:", files("CLIPLoaderGGUF", "clip_name"))
print("VAE:", files("VAELoader", "vae_name"))
print("LoRA:", files("LoraLoaderModelOnly", "lora_name"))

In [ ]:
# ── 6. Tunel ─────────────────────────────────────────────────────────────────
import subprocess, time, re, os

TLOG = "/content/tunnel.log"
subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True)
subprocess.run(f"nohup cloudflared tunnel --no-autoupdate --url http://127.0.0.1:8188 > {TLOG} 2>&1 &",
               shell=True, check=True)

url = None
for _ in range(60):
    time.sleep(2)
    txt = open(TLOG, errors="ignore").read() if os.path.exists(TLOG) else ""
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        url = m.group(0)
        break

if not url:
    print(open(TLOG, errors="ignore").read()[-1500:])
    raise RuntimeError("tunel nie wstał")

open("/content/tunnel.url", "w").write(url)
print("=" * 70)
print("COMFY_TUNNEL_URL =", url)
print("=" * 70)
print("Podaj ten URL Hermesowi — wejdzie po API i zbuduje workflow.")
print("UWAGA: tunel quick jest publiczny (bez auth). Kto zna URL, ten wchodzi.")

## Co dalej

1. Wklej `COMFY_TUNNEL_URL` do czatu — wchodzę po API (`/prompt`, `/history`, `/view`) i buduję flow: H3-Image na klatki → H3-Video I2V na ujęcia → FFmpeg concat.
2. Sesja Colaba musi żyć. Nie zamykaj karty; przy L4 to 1,94 CU/h.
3. Po skończonej pracy uruchom ostatnią komórkę — zabija ComfyUI i tunel, żeby nie palić jednostek.

### Uwagi techniczne
- H3-Image: 6 kroków, `euler` + `simple`, CFG 1 (brak negatywnego promptu — wszystko do pozytywu), natywnie 16 MP, edycja przez `<Picture 1>`…`<Picture 16>`.
- H3-Video: 768p short edge, 24 fps, do ~15 s, natywne stereo audio w jednym przebiegu; 2K wymaga osobnego upscalu.
- Na L4 (23 034 MiB = 22,49 GiB): DiT wideo int8 20,97 GB = 19,53 GiB, więc po `--reserve-vram 0.3` zostaje **2,66 GiB** na aktywacje — przy 768p wystarcza. Fallback: `SET = "l4-nsfw-gguf"` (DiT Q4_K_M 19,86 GB = 18,50 GiB → 3,70 GiB zapasu). Encoder (14,04 GiB) i DiT nie siedzą w VRAM jednocześnie — ComfyUI zwalnia encoder po conditioningu.
- Python 3.13.16 — ComfyUI i jego requirements mają wheels cp313, ale przy pierwszej instalce może zaciągnąć własne torch. Jeśli `pip` zgłosi konflikt, wystarczy `pip install -q -r requirements.txt` powtórzyć po restarcie runtime.
- Licencja: Community H3 wyklucza UE/UK/KR/USA — komercyjnie przez licencję Comfy.

In [ ]:
# ── 7. Stop — zwalnia GPU (uruchom, gdy kończysz) ────────────────────────────
import subprocess, os, signal, time

subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True)
if os.path.exists("/content/comfy.pid"):
    try:
        os.kill(int(open("/content/comfy.pid").read()), signal.SIGTERM)
        time.sleep(2)
        print("ComfyUI zatrzymane")
    except ProcessLookupError:
        print("ComfyUI już nie działało")
subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv", shell=True)
print("Możesz teraz rozłączyć sesję Colaba.")